### Explanation
This notebook shows how to buse a Nx20 sequence representation as input to esm2 in jax  <br>
To maintain gradients it uses a straith through estimator and a modified forward logic to bridge the gap of requiring a discrete seq. representation. <br>
Furthermore it converts esm2 dictionary to the colabdesign AA dictionary.

In [1]:
import jax
import jax.numpy as jnp
import equinox as eqx
import esm2quinox
from colabdesign.af.alphafold.common import residue_constants


# Mirroring the exact alphabet mapping provided
_alphabet = {
    "^": 0,
    ".": 1,
    "$": 2,
    "?": 3,
    "L": 4,
    "A": 5,
    "G": 6,
    "V": 7,
    "S": 8,
    "E": 9,
    "R": 10,
    "T": 11,
    "I": 12,
    "D": 13,
    "P": 14,
    "K": 15,
    "Q": 16,
    "N": 17,
    "F": 18,
    "Y": 19,
    "M": 20,
    "H": 21,
    "W": 22,
    "C": 23,
    "X": 24,
    "B": 25,
    "U": 26,
    "Z": 27,
    "O": 28,
    "-": 30,
    "1": 31,
    "#": 32,
}



In [2]:
esm2quinox._esm2._alphabet

{'^': 0,
 '.': 1,
 '$': 2,
 '?': 3,
 'L': 4,
 'A': 5,
 'G': 6,
 'V': 7,
 'S': 8,
 'E': 9,
 'R': 10,
 'T': 11,
 'I': 12,
 'D': 13,
 'P': 14,
 'K': 15,
 'Q': 16,
 'N': 17,
 'F': 18,
 'Y': 19,
 'M': 20,
 'H': 21,
 'W': 22,
 'C': 23,
 'X': 24,
 'B': 25,
 'U': 26,
 'Z': 27,
 'O': 28,
 '-': 30,
 '1': 31,
 '#': 32}

In [27]:
colab_aa_dict = residue_constants.restype_order
bindcraft_order = "".join(list(colab_aa_dict.keys()))
esm_indices = jnp.array([_alphabet[aa] for aa in bindcraft_order])

In [25]:
def forward_and_loss(logits, plm_model):
    # A. Safely extract weights directly from the LogitHead to avoid property errors
    full_embed_weights = plm_model.logit_head.linear2.weight



    # Map out the 20 structural amino acids, bridges the gap between two different dict of AA
    colab_aa_dict = residue_constants.restype_order
    bindcraft_order = "".join(list(colab_aa_dict.keys()))
    esm_indices = jnp.array([_alphabet[aa] for aa in bindcraft_order])
    aligned_weights = full_embed_weights[esm_indices, :]  # Shape: [20, 320]

    # Extract special context tokens
    cls_embed = full_embed_weights[_alphabet["^"], :]
    eos_embed = full_embed_weights[_alphabet["$"], :]

    # B. Softmax and Straight-Through Estimator (STE)
    p_softmax = jax.nn.softmax(logits, axis=-1)
    hard_indices = jnp.argmax(logits, axis=-1)
    hard_one_hot = jax.nn.one_hot(hard_indices, 20)
    p_ste = jax.lax.stop_gradient(hard_one_hot - p_softmax) + p_softmax

    # C. Map to continuous sequence embeddings
    seq_embeds = jnp.matmul(p_ste, aligned_weights)  # Shape: [10, 320]

    # D. Prepend <cls> and Append <eos>
    cls_expanded = jnp.expand_dims(cls_embed, (0,1))
    eos_expanded = jnp.expand_dims(eos_embed, (0,1))
    inputs_embeds = jnp.concatenate(
        [cls_expanded, seq_embeds, eos_expanded], axis=1
    )  # Shape: [12, 320]
    
    inputs_embeds =jnp.squeeze(inputs_embeds, axis=0)  # Shape: [320]
    # E. Replicate exact padding/dropout logic from _call
    # Since we design full-sequence targets, there are no interior pad tokens
    is_pad = jnp.zeros(
        (inputs_embeds.shape[0],), dtype=jnp.bool_
    )  # [12] containing False

    # F. Replicate the Exact Partition & Scan Loop from esm2quinox source!
    dynamic_layers, static_layer = eqx.partition(plm_model.layers, eqx.is_array)

    def f(x_carry, dynamic_layer):
        layer = eqx.combine(dynamic_layer, static_layer)
        x_out = layer(x_carry, is_pad=is_pad)  # Must feed is_pad array
        return x_out, None

    x, _ = jax.lax.scan(f, inputs_embeds, xs=dynamic_layers)

    # G. Apply original vmapped norms and prediction steps
    hidden = jax.vmap(plm_model.layer_norm)(x)
    plm_logits = jax.vmap(plm_model.logit_head)(hidden)  # Shape: [12, 33]

    # H. Strip context frames and isolate optimization target loss
    seq_plm_logits = plm_logits[1:-1, :]
    return jnp.mean(seq_plm_logits)


In [26]:
def test_jax_gradient_flow():
    print("Initializing JAX/Equinox environment...")

    key = jax.random.PRNGKey(42)
    model_key, logits_key = jax.random.split(key)

    # 1. Load the model topology
    model = esm2quinox.ESM2(
        num_layers=6, embed_size=320, num_heads=20, token_dropout=False, key=model_key
    )

    # 2. Continuous Logits from BindCraft optimization path: [Length=10, Vocab=20]
    dummy_logits = jax.random.normal(logits_key, (10, 20))
    
    dummy_logits = jnp.expand_dims(dummy_logits, axis=0)  # Shape: [1, 10, 20]

    print("Running forward pass and JAX autodiff...")
    loss_val, gradients = jax.value_and_grad(forward_and_loss, argnums=0)(dummy_logits, model)

    grad_magnitude = jnp.sum(jnp.abs(gradients))

    print("\n" + "=" * 40)
    print(f"Loss Value: {loss_val:.4f}")
    if grad_magnitude > 0:
        print(
            "🎉 SUCCESS: Gradient tracking perfectly matches internal source mechanics!"
        )
        print(f"Gradient Matrix Dimensions: {gradients.shape}")
        print(f"Absolute Gradient Accumulation: {grad_magnitude:.6f}")
    else:
        print("❌ ERROR: Gradient tracking graph fractured.")
    print("=" * 40)


if __name__ == "__main__":
    test_jax_gradient_flow()

Initializing JAX/Equinox environment...
Running forward pass and JAX autodiff...

Loss Value: 0.0734
🎉 SUCCESS: Gradient tracking perfectly matches internal source mechanics!
Gradient Matrix Dimensions: (1, 10, 20)
Absolute Gradient Accumulation: 0.132679


In [ ]:

key = jax.random.PRNGKey(42)
model_key, logits_key = jax.random.split(key)

# 1. Load the model topology
model = esm2quinox.ESM2(
    num_layers=6, embed_size=320, num_heads=20, token_dropout=False, key=model_key
)
model.logit_head